In [ ]:
%reload_ext sql
%sql sqlite:///../data/silver/metadata.db

In [58]:
%%sql
%%sql
SELECT
*
FROM recordings r
LIMIT 1

Running query in 'sqlite:///../data/silver/metadata.db'

id,status,rec,lat,lon,type,method,file,lic,q,auto,dvc,mic,smp,species,length_s
754598,identified,JAYRSON ARAUJO DE OLIVEIRA,-11.7519,-55.638,song,in the hand,https://xeno-canto.org/754598/download,https://creativecommons.org/licenses/by-nc-sa/4.0/,A,no,Panasonic RRUS551,Embutido,44100,Pyriglena similis,20


# 1. Impacto dos filtros de qualidade

In [ ]:
%%sql
%%sql
SELECT
COUNT(*)
FROM recordings r
WHERE r.status IS NOT "identified"

In [ ]:
%%sql
%%sql
SELECT
COUNT(*)
FROM recordings r
WHERE r.method == "in the hand"

In [ ]:
%%sql
%%sql
SELECT
COUNT(*)
FROM recordings r
WHERE r.type NOT LIKE '%song%'
AND r.type NOT LIKE '%canto%'
AND r.type NOT LIKE '%call%'
AND r.type NOT LIKE '%chamado%'

In [ ]:
%%sql
%%sql
SELECT
COUNT(*)
FROM recordings r
WHERE r.smp < 22050

In [ ]:
%%sql
%%sql
SELECT
COUNT(*)
FROM recordings r
WHERE r.length_s < 3

In [ ]:
%%sql
%%sql
SELECT
COUNT(*)
FROM recordings r
WHERE r.status IS NOT "identified"
OR r.method == "in the hand"
OR (r.type NOT LIKE '%song%'
AND r.type NOT LIKE '%canto%'
AND r.type NOT LIKE '%call%'
AND r.type NOT LIKE '%chamado%')
OR r.type IS NULL
OR r.smp < 22050
OR r.length_s < 3
OR r.file IS NULL

In [ ]:
%%sql
%%sql
SELECT
    r.species,
    COUNT(*) as total,
    (SUM(CASE WHEN r.method == "in the hand" THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) as pct_perdidas
FROM recordings r
GROUP BY r.species
HAVING COUNT(*) > 50
ORDER BY pct_perdidas DESC
LIMIT 20

- 7723 gravações não passam no filtro
- 5622 foram feitas na mão, maior parte, porém esse método está bem distribuído entre as espécies

# 2. Seleção espécies candidatas

In [ ]:
%%sql
%%sql
CREATE VIEW usable_recordings AS
SELECT *
FROM recordings r
WHERE r.status = 'identified'
  AND r.file IS NOT NULL
  AND r.method IS NOT NULL
  AND r.method <> 'in the hand'
  AND r.type IS NOT NULL
  AND (
        r.type LIKE '%song%'
        OR r.type LIKE '%canto%'
        OR r.type LIKE '%call%'
        OR r.type LIKE '%chamado%'
      )
  AND (r.smp IS NULL OR r.smp >= 22050)
  AND r.length_s >= 3

In [ ]:
%%sql
%%sql
SELECT
COUNT(*)
FROM (
SELECT
COUNT(*) as total,
COUNT(DISTINCT ur.rec) as num_gravadores
FROM usable_recordings ur
GROUP BY ur.species
HAVING total > 50 AND num_gravadores > 20) t

In [ ]:
%%sql
%%sql
SELECT
COUNT(*)
FROM (
SELECT
COUNT(*) as total,
COUNT(DISTINCT ur.rec) as num_gravadores
FROM usable_recordings ur
GROUP BY ur.species
HAVING total > 100 AND num_gravadores > 50) t

In [ ]:
%%sql
ATTACH DATABASE "../data/gold/gold.db" AS gold

In [53]:
%%sql
CREATE TEMP VIEW usable_in_forest AS
SELECT
    u.*,
    f.in_atlantic_forest
FROM usable_recordings u
LEFT JOIN gold.in_forest f
ON u.id = f.id

Running query in 'sqlite:///../data/silver/metadata.db'

RuntimeError: (sqlite3.OperationalError) view usable_in_forest already exists
[SQL: CREATE TEMP VIEW usable_in_forest AS
SELECT
    u.*,
    f.in_atlantic_forest
FROM usable_recordings u
LEFT JOIN gold.in_forest f
ON u.id = f.id]
(Background on this error at: https://sqlalche.me/e/21/e3q8)


In [52]:
%%sql
SELECT
    in_atlantic_forest,
    COUNT(*)
FROM usable_in_forest
GROUP BY in_atlantic_forest

Running query in 'sqlite:///../data/silver/metadata.db'

in_atlantic_forest,COUNT(*)
None,1479
0,27001
1,24349


In [57]:
%%sql
WITH por_especie AS (
    SELECT
        uif.species,
        SUM(CASE WHEN uif.in_atlantic_forest = 1 THEN 1 ELSE 0 END) AS grav_dentro,
        COUNT(DISTINCT uif.rec) AS aut_brasil
    FROM usable_in_forest uif
    GROUP BY uif.species
)
SELECT COUNT(*)
FROM por_especie
WHERE grav_dentro >= 100
AND aut_brasil >= 40

Running query in 'sqlite:///../data/silver/metadata.db'

COUNT(*)
39


In [54]:
%%sql
WITH por_especie AS (
    SELECT
        uif.species,
        SUM(CASE WHEN uif.in_atlantic_forest = 1 THEN 1 ELSE 0 END) AS grav_dentro,
        COUNT(*) AS grav_brasil,
        COUNT(DISTINCT uif.rec) AS aut_brasil,
        SUM(length_s) AS tempo_gravacao
    FROM usable_in_forest uif
    GROUP BY uif.species
)
SELECT *
FROM por_especie
WHERE grav_dentro >= 100
AND aut_brasil >= 40

Running query in 'sqlite:///../data/silver/metadata.db'

species,grav_dentro,grav_brasil,aut_brasil,tempo_gravacao
Attila rufus,136,138,59,6360
Automolus leucophthalmus,124,141,61,5469
Batara cinerea,111,112,49,4199
Campylorhamphus falcularius,105,113,53,5061
Cantorchilus longirostris,110,197,55,7326
Chiroxiphia caudata,183,192,77,9923
Conopophaga lineata,118,134,50,4399
Conopophaga melanops,112,112,49,4389
Cyclarhis gujanensis,130,225,74,7318
Drymophila ferruginea,148,157,50,7049


### Espécies selecionadas:
- 39 espécies, 5.750 gravações
- Critério: +100 gravações dentro da mata atlântica, +40 gravadores distintos por todo o brasil -> serão usadas gravações que não estão dentro da mata atlântica da espécie, ganho de mais gravações, as espécies não deixam de estar em maioridade na mata atlântica
- um mínimo de 40 autores distintos = um mínimo de 8 autores diferentes por ave para o teste da rede (em um split 80/20)

# 3. Espécies selecionadas estão no fundo da gravação de outras espécies selecionadas?

In [59]:
%%sql
SELECT
*
FROM background_species
LIMIT 1

Running query in 'sqlite:///../data/silver/metadata.db'

recording_id,species
754598,Tangara chilensis


In [118]:
%%sql
WITH
por_especie AS (
    SELECT
        uif.species,
        SUM(CASE WHEN uif.in_atlantic_forest = 1 THEN 1 ELSE 0 END) AS grav_dentro,
        COUNT(DISTINCT uif.rec) AS aut_brasil
    FROM usable_in_forest uif
    GROUP BY uif.species
),
candidatas AS (
    SELECT
        species
    FROM por_especie
    WHERE grav_dentro >= 100 AND aut_brasil >= 40
),
candidata_fundo AS (
    SELECT
        bgs.recording_id as id,
        bgs.species as background
    FROM background_species bgs
    WHERE bgs.species IN (SELECT species FROM candidatas)
),
perdidas AS (
    SELECT
        uif.id,
        uif.species,
        uif.rec,
        cf.background
    FROM usable_in_forest uif
    INNER JOIN candidata_fundo cf
    ON uif.id = cf.id AND uif.species != cf.background
    WHERE uif.species IN (SELECT species FROM candidatas)
),
grav_candidatas AS (
    SELECT
        uif.id,
        uif.species,
        uif.rec
    FROM usable_in_forest uif
    WHERE uif.species IN (SELECT species FROM candidatas)
)
SELECT
    gc.species,
    COUNT(DISTINCT gc.id) as total,
    COUNT(DISTINCT p.id) as would_lost,
    COUNT(DISTINCT CASE WHEN p.id IS NULL THEN gc.rec END) AS gravadores_restantes,
    COUNT(DISTINCT gc.rec) - COUNT(DISTINCT CASE WHEN p.id IS NULL THEN gc.rec END) AS gravadores_perdidos
FROM grav_candidatas gc
LEFT JOIN perdidas p
ON gc.id = p.id
GROUP BY gc.species


Running query in 'sqlite:///../data/silver/metadata.db'

species,total,would_lost,gravadores_restantes,gravadores_perdidos
Attila rufus,138,25,55,4
Automolus leucophthalmus,141,25,55,6
Batara cinerea,112,23,43,6
Campylorhamphus falcularius,113,19,49,4
Cantorchilus longirostris,197,13,50,5
Chiroxiphia caudata,192,24,75,2
Conopophaga lineata,134,24,46,4
Conopophaga melanops,112,20,42,7
Cyclarhis gujanensis,225,18,72,2
Drymophila ferruginea,157,31,43,7


### Decisão:
#### Gravações de espécies de selecionadas com outras espécies selecionadas ao fundo serão removidas
- 924 das 5750 gravações possuem uma das outra espécies de fundo
- Essas 924 estão bem distribuídas entre as espécies e autores, remove-las não faria com que uma espécie tivesse muitas gravações perdidas ou uma menor distribuição da gravação entre autores